In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 📊 RAG Evaluator Dashboard

## What We're Building

A comprehensive evaluation framework for RAG pipelines that measures:
- **Retrieval quality** — Are we fetching the right documents?
- **Answer groundedness** — Is the answer supported by retrieved context?
- **Answer correctness** — Does the answer match the expected answer?
- **Latency** — How fast is each pipeline stage?

## Why Evaluate RAG?

```
Without evaluation:  "It seems to work"  →  Ships broken RAG to production
With evaluation:     Precision=0.6, Groundedness=0.85  →  Know exactly what to fix
```

## Evaluation Metrics

| Metric | Measures | Range |
|--------|----------|-------|
| Precision@k | % of retrieved docs that are relevant | 0-1 |
| Recall@k | % of relevant docs that were retrieved | 0-1 |
| MRR | How early the first relevant doc appears | 0-1 |
| Groundedness | % of answer claims supported by context | 0-1 |
| Answer Correctness | Semantic match to reference answer | 0-1 |
| Latency | Time in seconds for each stage | 0+ |

## Stack
- **LangChain + ChromaDB** — RAG pipeline
- **Ollama** — LLM-as-judge for quality metrics
- **Custom evaluation harness** — no external eval framework needed

## Step 1 — Imports & Setup

In [ ]:
import time
import json
from dataclasses import dataclass, field, asdict
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

## Step 2 — Build a RAG Pipeline to Evaluate

We'll create a standard RAG system with a knowledge base and
a benchmark test set with ground-truth answers.

In [ ]:
# Knowledge base about a fictional company
kb_docs = [
    Document(page_content="NovaTech was founded in 2018 by Alice Zhang and Bob Martinez in San Francisco. The company started in a garage with $50,000 in savings.", metadata={"id": "d1", "topic": "history"}),
    Document(page_content="NovaTech's main product is CloudSync, a real-time data synchronization platform. It supports PostgreSQL, MongoDB, and Redis. CloudSync has 99.99% uptime SLA.", metadata={"id": "d2", "topic": "product"}),
    Document(page_content="NovaTech raised $5M in Series A (2019), $25M in Series B (2021), and $80M in Series C (2023). Total funding is $110M. Lead investor in Series C was Sequoia Capital.", metadata={"id": "d3", "topic": "funding"}),
    Document(page_content="NovaTech has 450 employees as of 2024. The engineering team is 200 people. Offices in San Francisco (HQ), Austin, and London. Remote work is available for all roles.", metadata={"id": "d4", "topic": "company"}),
    Document(page_content="CloudSync pricing: Starter ($49/mo, 10GB), Professional ($199/mo, 100GB), Enterprise ($999/mo, unlimited). All plans include 24/7 support and 99.99% uptime guarantee.", metadata={"id": "d5", "topic": "pricing"}),
    Document(page_content="NovaTech's revenue was $12M in 2022, $28M in 2023, and is projected to reach $50M in 2024. The company became profitable in Q2 2024 for the first time.", metadata={"id": "d6", "topic": "financials"}),
    Document(page_content="NovaTech's main competitors are DataBridge (backed by Google Ventures, $200M funding) and SyncFlow (bootstrapped, profitable since 2020). NovaTech differentiates on real-time performance.", metadata={"id": "d7", "topic": "competition"}),
    Document(page_content="NovaTech uses a microservices architecture with Kubernetes. The tech stack includes Go for backend services, React for frontend, and Apache Kafka for event streaming.", metadata={"id": "d8", "topic": "technology"}),
    Document(page_content="NovaTech launched DataVault in 2024, an encrypted backup service. It uses AES-256 encryption and stores data across three geographic regions for disaster recovery.", metadata={"id": "d9", "topic": "product"}),
    Document(page_content="NovaTech's customer base includes 2,000 companies. Key customers: Stripe (payment sync), Airbnb (reservation sync), and Shopify (inventory sync). Enterprise retention rate is 96%.", metadata={"id": "d10", "topic": "customers"}),
]

embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(kb_docs, embeddings, collection_name="eval_kb")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

print(f"Knowledge base: {len(kb_docs)} documents indexed")

In [ ]:
# Benchmark test set — questions with ground-truth answers and relevant doc IDs
benchmark = [
    {
        "question": "Who founded NovaTech and when?",
        "reference_answer": "NovaTech was founded in 2018 by Alice Zhang and Bob Martinez.",
        "relevant_doc_ids": ["d1"],
    },
    {
        "question": "What is CloudSync's uptime SLA?",
        "reference_answer": "CloudSync has a 99.99% uptime SLA.",
        "relevant_doc_ids": ["d2", "d5"],
    },
    {
        "question": "How much total funding has NovaTech raised?",
        "reference_answer": "NovaTech has raised $110M in total funding across three rounds.",
        "relevant_doc_ids": ["d3"],
    },
    {
        "question": "What is the Enterprise plan price for CloudSync?",
        "reference_answer": "The Enterprise plan costs $999 per month with unlimited storage.",
        "relevant_doc_ids": ["d5"],
    },
    {
        "question": "When did NovaTech become profitable?",
        "reference_answer": "NovaTech became profitable for the first time in Q2 2024.",
        "relevant_doc_ids": ["d6"],
    },
    {
        "question": "What tech stack does NovaTech use?",
        "reference_answer": "NovaTech uses Go for backend, React for frontend, Apache Kafka for events, running on Kubernetes microservices.",
        "relevant_doc_ids": ["d8"],
    },
    {
        "question": "Name some key NovaTech customers",
        "reference_answer": "Key customers include Stripe, Airbnb, and Shopify.",
        "relevant_doc_ids": ["d10"],
    },
    {
        "question": "What encryption does DataVault use?",
        "reference_answer": "DataVault uses AES-256 encryption and stores data across three geographic regions.",
        "relevant_doc_ids": ["d9"],
    },
]

print(f"Benchmark: {len(benchmark)} test questions with ground truth")

## Step 3 — RAG Pipeline with Instrumentation

In [ ]:
@dataclass
class RAGResult:
    """Captures all data from a single RAG invocation."""
    question: str
    answer: str
    retrieved_doc_ids: list[str]
    retrieved_texts: list[str]
    retrieval_latency_ms: float
    generation_latency_ms: float
    total_latency_ms: float


answer_prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the provided context.
Be specific and concise.

Context:
{context}

Question: {question}

Answer:"""
)


def instrumented_rag(question: str) -> RAGResult:
    """RAG pipeline with timing instrumentation."""
    # Retrieval
    t0 = time.perf_counter()
    docs = retriever.invoke(question)
    retrieval_ms = (time.perf_counter() - t0) * 1000

    context = "\n\n".join(d.page_content for d in docs)
    doc_ids = [d.metadata.get("id", "unknown") for d in docs]

    # Generation
    t1 = time.perf_counter()
    chain = answer_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    generation_ms = (time.perf_counter() - t1) * 1000

    return RAGResult(
        question=question,
        answer=answer,
        retrieved_doc_ids=doc_ids,
        retrieved_texts=[d.page_content for d in docs],
        retrieval_latency_ms=round(retrieval_ms, 1),
        generation_latency_ms=round(generation_ms, 1),
        total_latency_ms=round(retrieval_ms + generation_ms, 1),
    )


# Test
result = instrumented_rag("Who founded NovaTech?")
print(f"Answer: {result.answer}")
print(f"Docs: {result.retrieved_doc_ids}")
print(f"Latency: retrieval={result.retrieval_latency_ms}ms, generation={result.generation_latency_ms}ms")

## Step 4 — Evaluation Metrics

In [ ]:
# --- Retrieval Metrics ---

def precision_at_k(retrieved_ids: list[str], relevant_ids: list[str]) -> float:
    """What fraction of retrieved docs are relevant?"""
    if not retrieved_ids:
        return 0.0
    relevant_set = set(relevant_ids)
    hits = sum(1 for doc_id in retrieved_ids if doc_id in relevant_set)
    return hits / len(retrieved_ids)


def recall_at_k(retrieved_ids: list[str], relevant_ids: list[str]) -> float:
    """What fraction of relevant docs were retrieved?"""
    if not relevant_ids:
        return 1.0
    relevant_set = set(relevant_ids)
    hits = sum(1 for doc_id in retrieved_ids if doc_id in relevant_set)
    return hits / len(relevant_set)


def mrr(retrieved_ids: list[str], relevant_ids: list[str]) -> float:
    """Reciprocal rank of first relevant document."""
    relevant_set = set(relevant_ids)
    for i, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant_set:
            return 1.0 / i
    return 0.0


print("Retrieval metrics defined: precision_at_k, recall_at_k, mrr")

In [ ]:
# --- LLM-as-Judge Metrics ---

groundedness_prompt = ChatPromptTemplate.from_template(
    """Rate how well the answer is supported by the provided context.

Context:
{context}

Answer:
{answer}

Score from 0 to 10:
- 0: Answer is completely unsupported / hallucinated
- 5: Partially supported, some claims lack evidence
- 10: Every claim in the answer is directly supported by context

Respond with ONLY a number from 0 to 10.
Score:"""
)

correctness_prompt = ChatPromptTemplate.from_template(
    """Rate how well the generated answer matches the reference answer.

Reference answer: {reference}
Generated answer: {answer}

Score from 0 to 10:
- 0: Completely wrong or irrelevant
- 5: Partially correct, missing key information
- 10: Captures all key facts from the reference

Respond with ONLY a number from 0 to 10.
Score:"""
)


def extract_score(response: str) -> float:
    """Parse a 0-10 score from LLM response."""
    import re
    match = re.search(r'\b(10|[0-9])\b', response)
    return int(match.group(1)) / 10.0 if match else 0.5


def score_groundedness(answer: str, context: str) -> float:
    chain = groundedness_prompt | llm | StrOutputParser()
    response = chain.invoke({"answer": answer, "context": context})
    return extract_score(response)


def score_correctness(answer: str, reference: str) -> float:
    chain = correctness_prompt | llm | StrOutputParser()
    response = chain.invoke({"answer": answer, "reference": reference})
    return extract_score(response)


print("LLM-as-Judge metrics defined: groundedness, correctness")

## Step 5 — Run Full Evaluation

In [ ]:
@dataclass
class EvalResult:
    question: str
    answer: str
    reference: str
    precision: float
    recall: float
    mrr_score: float
    groundedness: float
    correctness: float
    retrieval_ms: float
    generation_ms: float
    total_ms: float


def evaluate_benchmark(benchmark_data: list[dict]) -> list[EvalResult]:
    """Run full evaluation on benchmark dataset."""
    results = []

    for i, item in enumerate(benchmark_data, 1):
        print(f"\n[{i}/{len(benchmark_data)}] {item['question'][:50]}...")

        # Run RAG
        rag_result = instrumented_rag(item["question"])

        # Retrieval metrics
        prec = precision_at_k(rag_result.retrieved_doc_ids, item["relevant_doc_ids"])
        rec = recall_at_k(rag_result.retrieved_doc_ids, item["relevant_doc_ids"])
        mrr_val = mrr(rag_result.retrieved_doc_ids, item["relevant_doc_ids"])

        # LLM-as-judge metrics
        context = "\n\n".join(rag_result.retrieved_texts)
        ground = score_groundedness(rag_result.answer, context)
        correct = score_correctness(rag_result.answer, item["reference_answer"])

        eval_result = EvalResult(
            question=item["question"],
            answer=rag_result.answer,
            reference=item["reference_answer"],
            precision=prec,
            recall=rec,
            mrr_score=mrr_val,
            groundedness=ground,
            correctness=correct,
            retrieval_ms=rag_result.retrieval_latency_ms,
            generation_ms=rag_result.generation_latency_ms,
            total_ms=rag_result.total_latency_ms,
        )
        results.append(eval_result)

        print(f"  P@3={prec:.2f} R@3={rec:.2f} MRR={mrr_val:.2f} Ground={ground:.1f} Correct={correct:.1f} Time={rag_result.total_latency_ms:.0f}ms")

    return results


print("Running evaluation on benchmark...")
eval_results = evaluate_benchmark(benchmark)

## Step 6 — Dashboard Summary

In [ ]:
def print_dashboard(results: list[EvalResult]) -> None:
    """Print a comprehensive evaluation dashboard."""
    n = len(results)

    # Aggregate metrics
    avg_precision = sum(r.precision for r in results) / n
    avg_recall = sum(r.recall for r in results) / n
    avg_mrr = sum(r.mrr_score for r in results) / n
    avg_groundedness = sum(r.groundedness for r in results) / n
    avg_correctness = sum(r.correctness for r in results) / n
    avg_retrieval_ms = sum(r.retrieval_ms for r in results) / n
    avg_generation_ms = sum(r.generation_ms for r in results) / n
    avg_total_ms = sum(r.total_ms for r in results) / n

    print("\n" + "=" * 60)
    print("📊 RAG EVALUATION DASHBOARD")
    print("=" * 60)

    print(f"\n📦 Benchmark: {n} questions")
    print(f"\n--- Retrieval Quality ---")
    print(f"  Precision@3:    {avg_precision:.2f}  {'✅' if avg_precision >= 0.5 else '⚠️'}")
    print(f"  Recall@3:       {avg_recall:.2f}  {'✅' if avg_recall >= 0.7 else '⚠️'}")
    print(f"  MRR:            {avg_mrr:.2f}  {'✅' if avg_mrr >= 0.7 else '⚠️'}")

    print(f"\n--- Answer Quality ---")
    print(f"  Groundedness:   {avg_groundedness:.2f}  {'✅' if avg_groundedness >= 0.7 else '⚠️'}")
    print(f"  Correctness:    {avg_correctness:.2f}  {'✅' if avg_correctness >= 0.7 else '⚠️'}")

    print(f"\n--- Latency ---")
    print(f"  Retrieval:      {avg_retrieval_ms:.0f}ms avg")
    print(f"  Generation:     {avg_generation_ms:.0f}ms avg")
    print(f"  Total:          {avg_total_ms:.0f}ms avg")

    # Per-question breakdown
    print(f"\n--- Per-Question Breakdown ---")
    print(f"{'#':<3} {'P@3':>4} {'R@3':>4} {'MRR':>4} {'Grnd':>5} {'Corr':>5} {'ms':>6}  Question")
    print("-" * 70)
    for i, r in enumerate(results, 1):
        q_short = r.question[:30]
        print(f"{i:<3} {r.precision:>4.2f} {r.recall:>4.2f} {r.mrr_score:>4.2f} {r.groundedness:>5.2f} {r.correctness:>5.2f} {r.total_ms:>6.0f}  {q_short}")

    # Find weaknesses
    print(f"\n--- Potential Issues ---")
    weak_retrieval = [r for r in results if r.recall < 0.5]
    weak_grounded = [r for r in results if r.groundedness < 0.7]
    weak_correct = [r for r in results if r.correctness < 0.7]

    if weak_retrieval:
        print(f"  ⚠️ {len(weak_retrieval)} questions have low retrieval recall:")
        for r in weak_retrieval:
            print(f"     - {r.question[:50]}")
    if weak_grounded:
        print(f"  ⚠️ {len(weak_grounded)} answers have low groundedness:")
        for r in weak_grounded:
            print(f"     - {r.question[:50]}")
    if weak_correct:
        print(f"  ⚠️ {len(weak_correct)} answers have low correctness:")
        for r in weak_correct:
            print(f"     - {r.question[:50]}")
    if not (weak_retrieval or weak_grounded or weak_correct):
        print("  ✅ No major issues detected!")

    print("\n" + "=" * 60)


print_dashboard(eval_results)

## Step 7 — Compare Pipeline Variants

Evaluate different k values to find the optimal retrieval size.

In [ ]:
def quick_eval(k: int, benchmark_data: list[dict]) -> dict:
    """Quick evaluation with different k values."""
    local_retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    metrics = {"precision": [], "recall": [], "mrr": []}

    for item in benchmark_data:
        docs = local_retriever.invoke(item["question"])
        doc_ids = [d.metadata.get("id", "unknown") for d in docs]

        metrics["precision"].append(precision_at_k(doc_ids, item["relevant_doc_ids"]))
        metrics["recall"].append(recall_at_k(doc_ids, item["relevant_doc_ids"]))
        metrics["mrr"].append(mrr(doc_ids, item["relevant_doc_ids"]))

    return {
        "k": k,
        "avg_precision": sum(metrics["precision"]) / len(metrics["precision"]),
        "avg_recall": sum(metrics["recall"]) / len(metrics["recall"]),
        "avg_mrr": sum(metrics["mrr"]) / len(metrics["mrr"]),
    }


print("Comparing retrieval with different k values:\n")
print(f"{'k':>3}  {'Precision':>10}  {'Recall':>8}  {'MRR':>6}")
print("-" * 35)

for k in [1, 2, 3, 5, 7]:
    result = quick_eval(k, benchmark)
    print(f"{result['k']:>3}  {result['avg_precision']:>10.3f}  {result['avg_recall']:>8.3f}  {result['avg_mrr']:>6.3f}")

print("\n💡 Insight: Higher k increases recall but decreases precision.")
print("   Find the k where recall is high enough without drowning the LLM in noise.")

## Step 8 — Export Results

In [ ]:
# Export evaluation results for tracking over time
export = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model": "qwen3.5:9b",
    "embedding": "nomic-embed-text-v2-moe",
    "k": 3,
    "num_questions": len(eval_results),
    "aggregate": {
        "avg_precision": round(sum(r.precision for r in eval_results) / len(eval_results), 3),
        "avg_recall": round(sum(r.recall for r in eval_results) / len(eval_results), 3),
        "avg_mrr": round(sum(r.mrr_score for r in eval_results) / len(eval_results), 3),
        "avg_groundedness": round(sum(r.groundedness for r in eval_results) / len(eval_results), 3),
        "avg_correctness": round(sum(r.correctness for r in eval_results) / len(eval_results), 3),
    },
    "per_question": [asdict(r) for r in eval_results]
}

print(json.dumps(export["aggregate"], indent=2))
print("\n✅ Results ready for export. Save to file to track improvements over time.")

## 🧠 Key Concepts Recap

| Metric Type | Metrics | What They Tell You |
|-------------|---------|--------------------|
| **Retrieval** | Precision@k, Recall@k, MRR | Are we finding the right docs? |
| **Generation** | Groundedness | Is the answer backed by evidence? |
| **End-to-end** | Correctness | Does the answer match ground truth? |
| **Performance** | Latency (ms) | Is it fast enough for users? |

## When Each Metric Is Low

| Problem | Symptom | Fix |
|---------|---------|-----|
| Low precision | Irrelevant docs retrieved | Better embeddings, reranking |
| Low recall | Missing relevant docs | Increase k, query expansion |
| Low groundedness | LLM hallucinating | Better prompts, lower temperature |
| Low correctness | Wrong answers | Better retrieval + better prompts |
| High latency | Slow responses | Smaller model, caching, fewer docs |

## 🔧 Production Extensions

- **LangSmith**: Cloud-hosted tracing and evaluation
- **RAGAS**: Open-source RAG evaluation framework
- **Continuous eval**: Run benchmarks on every deployment
- **A/B testing**: Compare pipeline variants on live traffic
- **Human eval**: Sample answers for human rating alongside LLM-as-judge